# 12.12 · 检索增强生成 / Retrieval-Augmented Generation (RAG)

> **课程定位 / Where this fits**
> 第 12 课，**Part 12**。让 LLM "查着资料回答"——工业界落地 LLM 最重要的模式之一。
> Lesson 12, **Part 12**. Letting an LLM "answer with references" — one of the most important production LLM patterns.
>
> LLM 有三个硬伤：①**知识冻结**在训练时刻(不知道最新事件)；②**幻觉**——不知道却"一本正经地编造"；③**没有你的私有/内部数据**。**RAG(检索增强生成)** 优雅地解决这些：把外部知识(文档/数据库/网页)建成**可检索的索引**，回答前先**检索出相关内容**，再把它塞进 prompt 让 LLM **基于检索到的事实作答**。不用重训模型、知识可随时更新、答案有出处、大幅减少幻觉。本课**从零搭一个 RAG 检索流程**。
> LLMs have three hard limits: ① **frozen knowledge** (no recent events); ② **hallucination** — confidently making things up; ③ **no access to your private/internal data**. **RAG** elegantly fixes these: build external knowledge (docs/DB/web) into a **searchable index**, **retrieve relevant content** before answering, and stuff it into the prompt so the LLM **answers from retrieved facts**. No retraining, knowledge updatable anytime, answers with sources, far less hallucination. We **build a RAG retrieval pipeline from scratch**.
>
> 💼 **实战/面试视角**：RAG 是 LLM 落地**最常见**的架构——"RAG 流程 / 为什么用 RAG 而非微调 / chunking / 检索质量 / reranker / 混合检索" 几乎必问。
> 💼 **Practical/interview angle:** RAG is the **most common** production LLM architecture — "RAG pipeline / RAG vs fine-tuning / chunking / retrieval quality / reranker / hybrid search" almost always asked.

> 📐 **符号约定 / Notation**
> - chunk —— 文档切成的小段(检索单位) / a document chunk (retrieval unit)
> - 嵌入(embedding) —— 把文本变成向量 / text → vector
> - top-k 检索 —— 取与查询最相似的 k 个 chunk / retrieve the k nearest chunks

> 💡 **面试相关 / Interview-relevant**
> - "RAG 的完整流程(索引→检索→增强→生成)"（出镜率 ★★★★★）
> - "什么时候用 RAG 而不是微调"（★★★★★）
> - "chunking 策略 / chunk 大小权衡"（★★★★）
> - "怎么提升检索质量(reranker/混合检索/query改写)"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 RAG 解决 LLM 的哪些硬伤、何时用。
   Understand which LLM limits RAG fixes and when to use it.
2. 掌握 RAG 四步流程: 索引→检索→增强→生成。
   Master RAG's four steps: index → retrieve → augment → generate.
3. **从零实现检索**(嵌入 + 余弦 top-k)。
   Implement retrieval from scratch (embedding + cosine top-k).
4. 看到检索如何"接地(grounding)"答案、减少幻觉。
   See how retrieval "grounds" answers and reduces hallucination.

## 目录 / TOC
1. [为什么要 RAG ⭐](#1)
2. [RAG 四步流程 ⭐](#2)
3. [从零实现检索 ⭐](#3)
4. [接地生成、提升检索 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么要 RAG ⭐ / Why RAG

LLM 把知识"压进"了权重，但这带来问题：
LLMs compress knowledge into weights, causing problems:
- **知识冻结**：模型只知道训练截止前的事，问它"今天的新闻/上周的财报"它不知道。
  **Frozen knowledge:** knows only up to its training cutoff; can't answer "today's news / last week's report."
- **幻觉(hallucination)**：不知道答案时，它仍会**流畅地编造**一个看似可信的错误答案——这在企业应用里是大忌。
  **Hallucination:** when it doesn't know, it still **fluently fabricates** a plausible-but-wrong answer — unacceptable in enterprise apps.
- **没有私有数据**：你公司的内部文档、产品手册、个人笔记，模型从没见过。
  **No private data:** your company docs, manuals, personal notes — never seen.

**微调能解决吗？** 不理想：微调贵、知识更新就得重训、且仍可能幻觉、难追溯出处。**RAG 是更好的选择**：知识放在**外部可检索库**里，模型回答时**现查现用**——更新知识只需更新库(秒级)、答案有出处可核查、几乎不需训练。
**Could fine-tuning fix it?** Not ideal: expensive, needs retraining to update, still hallucinates, hard to cite. **RAG is better**: knowledge lives in an **external searchable store**, looked up at answer time — updating is instant, answers are citable, almost no training.

> **RAG vs 微调(面试)**：知识/事实类(尤其常变、私有)→**RAG**；改变模型的**行为/风格/格式**→微调。常常**两者结合**。
> **RAG vs fine-tuning:** knowledge/facts (esp. changing/private) → **RAG**; changing **behavior/style/format** → fine-tuning. Often **combined**.


<a id="2"></a>
## 2. RAG 四步流程 ⭐ / RAG's Four Steps

RAG 系统分**离线建库**和**在线问答**两阶段，共四步(面试要能完整说出)：
A RAG system has **offline indexing** and **online answering**, four steps total (be able to recite):
1. **索引(Index, 离线)**：把知识文档**切成小块(chunk)**，每块用**嵌入模型**转成向量，存进**向量数据库**(12.13)。
   **Index (offline):** split docs into **chunks**, embed each with an **embedding model** into a vector, store in a **vector DB** (12.13).
2. **检索(Retrieve, 在线)**：把用户问题也**嵌入成向量**，在库里找**最相似的 top-k 个 chunk**(余弦相似度)。
   **Retrieve (online):** embed the user's question, find the **top-k most similar chunks** (cosine similarity).
3. **增强(Augment)**：把检索到的 chunk **拼进 prompt**，作为"参考资料"。
   **Augment:** stuff retrieved chunks into the **prompt** as "reference material."
4. **生成(Generate)**：LLM **基于这些参考资料**回答(并可附出处)。
   **Generate:** the LLM **answers based on those references** (and can cite sources).

核心 prompt 模板：
The core prompt template:
```
根据以下参考资料回答问题。如果资料里没有, 就说"不知道"(别编)。
参考资料:
{检索到的 top-k chunks}
问题: {用户问题}
回答:
```
"如果没有就说不知道"这句话是**抑制幻觉**的关键。
The "say 'I don't know' if not present" line is key to **suppressing hallucination**.


<a id="3"></a>
## 3. 从零实现检索 ⭐ / Implementing Retrieval From Scratch

检索是 RAG 的核心。我们从零搭：建一个小知识库 → 把每个 chunk 嵌入成向量(这里用 **TF-IDF** 作为嵌入，呼应 11.2；真实 RAG 用 Sentence-BERT 等**稠密语义嵌入**) → 对查询取**余弦相似度 top-k**。
Retrieval is RAG's core. From scratch: build a small knowledge base → embed each chunk (we use **TF-IDF**, echoing 11.2; real RAG uses **dense semantic embeddings** like Sentence-BERT) → retrieve **top-k by cosine similarity** for a query.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
sns.set_theme(style="whitegrid")

# 小知识库(假设是"某公司内部文档"的 chunk) / a small knowledge base (company-doc chunks)
knowledge = [
    "The company was founded in 2015 by Alice Chen in Singapore.",
    "Our flagship product, NovaCloud, is a data analytics platform launched in 2019.",
    "NovaCloud pricing starts at 49 dollars per month for the basic plan.",
    "The engineering team is based in Berlin and has about 120 employees.",
    "Customer support is available 24/7 via chat and email in English and Mandarin.",
    "The refund policy allows full refunds within 30 days of purchase.",
    "NovaCloud integrates with Slack, Salesforce, and Google Sheets.",
    "The company raised a 50 million dollar Series B round in 2022.",
]
# 索引: 把每个 chunk 嵌入成向量 / index: embed each chunk
embedder = TfidfVectorizer(stop_words="english")
index = embedder.fit_transform(knowledge)                 # (n_chunks, vocab) 向量库 / the vector index
print(f"知识库: {len(knowledge)} 个 chunk, 每个嵌入成 {index.shape[1]} 维向量")

def retrieve(query, k=2):
    q = embedder.transform([query])                       # 查询也嵌入成向量 / embed the query
    sims = cosine_similarity(q, index)[0]                 # 与所有 chunk 的余弦相似度 / cosine to all chunks
    top = sims.argsort()[::-1][:k]                        # 取最相似的 k 个 / top-k
    return [(knowledge[i], sims[i]) for i in top]

for query in ["What is the NovaCloud pricing per month?", "Where is the engineering team based?", "What is the refund policy?"]:
    print(f"\n问: {query}")
    for chunk, score in retrieve(query):
        print(f"  [相似度 {score:.2f}] {chunk}")
print("\n检索: 把问题嵌入→和库里每个chunk算余弦相似度→取最相似的→这些就是'参考资料'")

# ⚠️ 诚实的失败例子: TF-IDF 只看词面, 同义改写会检索错 / honest failure: TF-IDF is lexical, paraphrase breaks it
q_fail = "How much does NovaCloud cost?"
print(f"\n⚠️ 词面不匹配的失败例子: {q_fail}")
for chunk, score in retrieve(q_fail, k=1):
    print(f"  [相似度 {score:.2f}] {chunk}  ← 检索错了!")
print("  问题问'cost', 但价格 chunk 里写的是'pricing/dollars/per month'——词不一样, TF-IDF 匹配失败")
print("  → 这正是为什么真实 RAG 用'稠密语义嵌入'(能理解 cost≈pricing≈价格), 而非只看词面的 TF-IDF")


<a id="4"></a>
## 4. 接地生成、提升检索 + 小结 ⭐ / Grounded Generation & Improving Retrieval

把检索到的 chunk 拼进 prompt，LLM 就能**基于事实回答**。下面模拟完整 RAG 流程(用一个简单的抽取式"生成器"代替真实 LLM——本机无 LLM；真实 RAG 这一步是 LLM 阅读参考资料后作答)。重点看：**有检索 = 答案有据可查；无检索 = 只能靠模型记忆(可能幻觉)**。
Stuffing retrieved chunks into the prompt lets the LLM **answer from facts**. Below we simulate the full RAG flow (a simple extractive "generator" stands in for a real LLM — none here; in real RAG this step is an LLM reading the references). Key contrast: **with retrieval = grounded answer; without = rely on model memory (may hallucinate)**.


In [ ]:
def rag_answer(query, k=2):
    retrieved = retrieve(query, k)                        # 1-2步: 检索 / retrieve
    context = "\n".join(f"- {c}" for c, _ in retrieved)   # 3步: 增强(拼进prompt) / augment
    prompt = (f"根据以下参考资料回答问题, 资料中没有就说'不知道'。\n"
              f"参考资料:\n{context}\n问题: {query}\n回答:")
    # 4步: 生成(这里用抽取式占位; 真实RAG是LLM阅读context作答) / generate (extractive stand-in)
    best_chunk = retrieved[0][0]
    return prompt, best_chunk

q = "What is the refund policy?"
prompt, grounded = rag_answer(q)
print("=== 喂给 LLM 的完整 prompt(含检索到的参考资料) ===")
print(prompt)
print(f"\n=== 接地的答案(基于检索到的事实) ===\n{grounded}")
print("\n对比: 无 RAG 时, 模型没见过这家公司 → 只能编造价格(幻觉); 有 RAG → 答案来自真实文档(可核查)")
print("注: 真实 RAG 第4步是 LLM 阅读参考资料后用自然语言作答; 这里用抽取式占位演示'接地'机制")


**怎么提升 RAG 效果**(面试加分，实战关键)：
**How to improve RAG** (interview bonus, crucial in practice):
- **chunking 策略**：chunk 太大→检索不精准、塞太多无关内容；太小→上下文破碎。常用几百 token + 重叠(overlap)。
  **Chunking:** too large → imprecise retrieval + irrelevant stuffing; too small → fragmented context. Typically a few hundred tokens with overlap.
- **更好的嵌入**：用**稠密语义嵌入**(Sentence-BERT/OpenAI embeddings)代替 TF-IDF，能匹配"语义相近但用词不同"(TF-IDF 只看词面)。
  **Better embeddings:** dense semantic embeddings (Sentence-BERT/OpenAI) over TF-IDF, matching meaning not just words.
- **重排序(reranker)**：先用快的向量检索召回 top-50，再用一个更强的**交叉编码器**精排出 top-5。两阶段=召回快+排序准。
  **Reranker:** retrieve top-50 fast by vectors, then a stronger **cross-encoder** reranks to top-5. Two-stage = fast recall + accurate ranking.
- **混合检索(hybrid)**：稠密向量(语义) + BM25(关键词, 见11.9)结合，兼顾语义和精确词匹配。
  **Hybrid search:** dense (semantic) + BM25 (keyword, 11.9), combining meaning and exact-term matching.
- **query 改写/扩展**：把口语化问题改写成更好的检索查询。
  **Query rewriting/expansion:** rephrase colloquial questions into better search queries.

```
RAG 解决: LLM知识冻结 + 幻觉 + 无私有数据; 不重训、知识可更新、答案有出处
四步: 索引(chunk+嵌入存库) → 检索(query嵌入取top-k余弦) → 增强(塞进prompt) → 生成(LLM据此答)
prompt关键: "资料中没有就说不知道" → 抑制幻觉
检索从零: TF-IDF/稠密嵌入 + 余弦 top-k; 真实用 Sentence-BERT 等稠密语义嵌入
RAG vs 微调: 知识/事实(常变/私有)用RAG; 行为/风格用微调; 常结合
提升: chunking + 好嵌入 + reranker重排 + 混合检索(稠密+BM25) + query改写
```

### 💡 面试速查 / Interview cheat-sheet
1. **RAG四步**: 索引(chunk+嵌入) → 检索(top-k余弦) → 增强(进prompt) → 生成。
   RAG: index (chunk+embed) → retrieve (top-k cosine) → augment → generate.
2. **解决什么**: 知识冻结/幻觉/私有数据; 答案接地+可更新+有出处。
   Fixes: frozen knowledge/hallucination/private data; grounded, updatable, citable.
3. **RAG vs 微调**: 事实知识用RAG, 行为风格用微调, 常结合。
   RAG vs fine-tune: facts → RAG, behavior → fine-tune, often combined.
4. **抑制幻觉**: prompt里要求"没有就说不知道"+只据检索内容答。
   Suppress hallucination: instruct "say I don't know" + answer only from retrieved content.
5. **提升检索**: chunking/稠密嵌入/reranker/混合检索(BM25+dense)/query改写。
   Improve retrieval: chunking/dense embeddings/reranker/hybrid/query rewriting.

### 下一节 / Next
**12.13 向量数据库**——RAG 要在**上百万个向量**里快速找最近邻, 暴力比较太慢。**向量数据库**用 **近似最近邻(ANN)** 索引(如 HNSW)实现毫秒级检索。我们会**从零实现向量检索**并理解 ANN 的核心思想。
**12.13 Vector Databases** — RAG must find nearest neighbors among **millions of vectors**; brute force is too slow. **Vector databases** use **approximate nearest neighbor (ANN)** indexes (e.g. HNSW) for millisecond search. We'll implement vector search from scratch and grasp ANN's core idea.
